# Ray tracing

> Channels ray-traced with Sionna RT: tables at fixed points, paths moved to continuous positions, and the tracing itself.

In [ ]:
#| default_exp radio.ray_tracing

In [ ]:
#| hide
from nbdev.showdoc import *

Ray tracing gives `h`, so a ray-traced channel is a `Fading` (see `fading`) like any other. Channels at either
level use it, as in Sionna's tutorials: at the link level
([Link-level simulations with Sionna RT](https://nvlabs.github.io/sionna/phy/tutorials/notebooks/Link_Level_Simulations_with_RT.html))
and at the system level ([SYS meets RT](https://nvlabs.github.io/sionna/sys/tutorials/notebooks/SYS_Meets_RT.html)):

```python
fading = RayTracedPathsFading.from_npz('room_paths.npz', subcarriers=...)
SystemLevelChannel(fading=fading, per_model=SionnaErrorModel(), ...)   # the SINR at each subcarrier -> BLER
SionnaLinkChannel(fading=fading, ...)                                  # every bit, 5G NR, h at each subcarrier
```

* `RayTracedFading`: a table of `H[rx, tx]` between fixed points (discrete environments: the nodes are on them).
* `RayTracedPathsFading`: the paths ray-traced at anchor points, moved to the nodes' exact positions (continuous
  environments).
* `trace_paths`: traces those paths in a Sionna RT scene. Only it needs Sionna RT: trace once where Sionna RT runs
  (e.g. a GPU node), save the arrays, and use them anywhere with numpy.

The nodes move, so the fading is told where they are, `set_positions`: on the fading, or on the channel, which
passes it on.

In [ ]:
#| export
from __future__ import annotations

import math
import warnings
from typing import Mapping, Sequence

import numpy as np

from comm_core.core import NodeId
from comm_core.radio.fading import Fading

### Ray-traced tables

The coefficients between every pair of a set of positions, e.g. the CFR from Sionna RT's `paths.cfr(...)`, saved
as numpy arrays:

* `H[rx, tx]`: complex, `(N, N)` or `(N, N, num_subcarriers)`.
* `positions[i]`: the position of index `i`, e.g. a grid cell `(x, y)`.

Nodes move, so the table is told where they are (`set_positions`, e.g. at every step of an environment). A
transmission gets the `H[rx, tx]` of its nodes' positions. Two nodes at the same position have a perfect link.
With subcarriers, a transmission gets one coefficient, `√(mean |H|²)`: the bits are not spread over the subcarriers.

The coefficients must keep the path loss, so compute them with `paths.cfr(..., normalize=False)`. With
`normalize=True` every link has `|H|² = 1` (only the phases differ) and every link would be equally good. The table
warns when it is given such an array.

In [ ]:
#| export
def _key(position) -> tuple:
    return tuple(np.asarray(position).reshape(-1).tolist())


class RayTracedFading(Fading):
    "Coefficients ray-traced beforehand (e.g. Sionna RT) between a set of positions, looked up where the nodes are."

    def __init__(
        self,
        H, # Channel coefficients `H[rx, tx]` between the positions: (N, N) or (N, N, num_subcarriers), complex
        positions: Sequence, # The N positions (e.g. grid cells (x, y)), in the order of H's indices
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
    ):
        H = np.asarray(H)
        if H.ndim == 3:
            gain = (np.abs(H) ** 2).mean(axis=-1)
        elif H.ndim == 2:
            gain = np.abs(H) ** 2
        else:
            raise ValueError(f"H: (N, N) or (N, N, num_subcarriers), not {H.shape}")
        if gain.shape[0] != gain.shape[1] or gain.shape[0] != len(positions):
            raise ValueError(f"H is {H.shape[:2]} for {len(positions)} positions")
        self.gain = gain.astype(np.float64)
        self.H = (H if H.ndim == 2 else np.sqrt(self.gain)).astype(np.complex128)   # one coefficient per link
        self.positions = [_key(p) for p in positions]
        self.index = {p: i for i, p in enumerate(self.positions)}
        off = ~np.eye(len(self.positions), dtype=bool)
        if off.any() and np.allclose(self.gain[off], 1.0):
            warnings.warn("every link has |H|^2 = 1: the coefficients were normalized (paths.cfr(normalize=True)), "
                          "so the path loss is lost and every link is equally good; use normalize=False")
        self.node_positions: dict = {}
        self.set_positions(node_positions or {})

    @classmethod
    def from_files(
        cls,
        channel_path: str, # .npy of H[rx, tx]
        positions_path: str, # .npy of the positions
        **kwargs, # The other arguments of `RayTracedFading`
    ) -> RayTracedFading:
        "Load the arrays saved after ray tracing."
        return cls(np.load(channel_path), np.load(positions_path), **kwargs)

    def set_positions(
        self,
        positions: Mapping[NodeId, Sequence], # Position of (some of) the nodes, e.g. {'agent_0': (3, 5)}
    ) -> None:
        "Where the nodes are now."
        for node, p in positions.items():
            self.node_positions[node] = _key(p)

    def _index(self, node: NodeId) -> int:
        if node not in self.node_positions:
            raise KeyError(f"no position for node {node!r}: call set_positions first")
        p = self.node_positions[node]
        if p not in self.index:
            raise KeyError(f"node {node!r} is at {p}, where no channel was ray-traced ({len(self.index)} positions)")
        return self.index[p]

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        tx, rx = self._index(sender), self._index(receiver)
        return math.inf if tx == rx else float(self.gain[rx, tx])

    def gain_between(
        self,
        tx_position: Sequence, # A transmitter's position (one of the table's)
        rx_position: Sequence, # A receiver's position (one of the table's)
    ) -> float:
        "Linear path gain |H|^2 between two positions of the table, wherever the nodes are (inf if the same)."
        tx, rx = _key(tx_position), _key(rx_position)
        if tx not in self.index or rx not in self.index:
            raise KeyError(f"no channel was ray-traced between {tx} and {rx}")
        return math.inf if tx == rx else float(self.gain[self.index[rx], self.index[tx]])

    def coefficient(self, transmission, link):
        "H[rx, tx] at the nodes' current positions (math.inf if co-located)."
        tx, rx = self._index(transmission.sender), self._index(transmission.receiver)
        return math.inf if tx == rx else complex(self.H[rx, tx])

In [ ]:
show_doc(RayTracedFading.from_files)

In [ ]:
show_doc(RayTracedFading.set_positions)

In [ ]:
show_doc(RayTracedFading.path_gain)

In [ ]:
show_doc(RayTracedFading.gain_between)

In [ ]:
import tempfile, os

from comm_core.core import Transmission
from comm_core.radio.fading import NoFading

In [ ]:
t = Transmission(message=None, sender='a', receiver='b', start_time=0.0)

In [ ]:
positions = [(0, 0), (1, 0), (5, 0)]
H = np.array([[0, 1e-3, 2e-5], [1e-3j, 0, 3e-5], [2e-5, 3e-5, 0]])
table = RayTracedFading(H, positions, node_positions={'a': (0, 0), 'b': (1, 0)})
assert table.coefficient(t, None) == 1e-3j and abs(table.path_gain('a', 'b') - 1e-6) < 1e-18
assert abs(table.gain_between((0, 0), (1, 0)) - 1e-6) < 1e-18 and table.gain_between((5, 0), (5, 0)) == math.inf
table.set_positions({'b': (0, 0)})
assert table.coefficient(t, None) == math.inf and table.path_gain('a', 'b') == math.inf     # same cell
table.set_positions({'b': (9, 9)})
try:
    table.coefficient(t, None); raise AssertionError('no error')
except KeyError as e:
    assert 'no channel was ray-traced' in str(e)
H3 = np.repeat(H[..., None], 4, axis=-1)
assert np.allclose(RayTracedFading(H3, positions).gain, table.gain)
with tempfile.TemporaryDirectory() as tmp:
    np.save(os.path.join(tmp, 'H.npy'), H); np.save(os.path.join(tmp, 'pos.npy'), np.array(positions))
    assert RayTracedFading.from_files(os.path.join(tmp, 'H.npy'), os.path.join(tmp, 'pos.npy')).positions == positions
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    RayTracedFading(np.exp(1j * np.ones((3, 3))), positions)
    assert any('normalize' in str(x.message) for x in w)

### Ray-traced paths: continuous positions

A table of channels at fixed points (`RayTracedFading`) suits discrete environments, where the nodes are always
on one of them. In continuous environments (robots, vehicles), a node is almost never exactly at a ray-traced point.
The channel cannot simply be taken from the nearest point either: its phase turns a full circle every wavelength
(5.8 cm at 5.18 GHz), so the nearest point's channel would be wrong for anything coherent.

`RayTracedPathsFading` keeps the *paths* ray-traced at anchor points, e.g. every 5 cm (see `grid_points`), instead
of their sum. Each path has a gain `a`, a delay `τ`, and the directions in which it leaves the transmitter
(`u_tx`) and reaches the receiver (`u_rx`). A node a little away from its anchor sees the same paths, moved:

* **The path as a line.** Unfolded at its reflections, a path is a straight line of length `L = c τ` that reaches
  the receiver from an image of the transmitter, in the direction `u_rx`.
* **Moving the ends.** Moving the receiver by `Δ_rx` and the transmitter by `Δ_tx` moves the ends of that line.
  The transmitter's image moves by `Δ_tx` mirrored, so that `u_tx` turns into the arrival direction. This gives
  the new length `L'`.
* **The moved path.** Its delay is `τ' = L' / c`, and its gain is `a · L / L' · exp(−j 2π f_c (τ' − τ))` (spreading
  over the longer or shorter path, and its phase).

The channel response at frequency `f` from the carrier is then `H(f) = Σ a e^{−j 2π f τ'}`.

* **Accuracy:** this is exact for line-of-sight paths and single reflections off flat surfaces, wherever the
  ends move. Every other path (several reflections, diffraction) is right to first order: the plane-wave
  approximation, `Δτ = −(u_tx·Δ_tx + u_rx·Δ_rx) / c`. It stays accurate while the paths themselves (which
  walls they hit) don't change between anchors, so the anchors can be much further apart than a wavelength. The
  tests below and the validation against Sionna RT (at the end) measure it.
* **Moving ends:** both can move, e.g. agent-to-agent links, with the transmitter and receiver anchors on the same
  set of points. A fixed access point is a single transmitter anchor.
* **Close nodes:** nodes on the same anchor (nearer each other than the anchors) have a perfect link, like nodes
  at the same place.
* **Direction:** links go both ways (`reciprocal`): a robot → access point link uses the access point → robot
  paths in reverse.
* **Data:** `trace_paths` (below) traces the anchors in a Sionna RT scene, and returns the arguments of
  `RayTracedPathsFading`.

In [ ]:
#| export
def grid_points(
    lo: Sequence[float], # Lower corner: (x, y) or (x, y, z)
    hi: Sequence[float], # Upper corner (included when the spacing divides the size)
    spacing: float, # Distance between neighbouring points, e.g. 0.05 (m)
    height: float | None = None, # z of the points of a 2-D grid (the antennas' height)
) -> np.ndarray:
    "Anchor points on a regular grid: (N, 3)."
    lo, hi = np.asarray(lo, float), np.asarray(hi, float)
    if lo.shape != hi.shape or lo.shape[0] not in (2, 3):
        raise ValueError("lo and hi: both (x, y) or both (x, y, z)")
    axes = [np.arange(l, h + spacing / 2, spacing) for l, h in zip(lo, hi)]
    points = np.stack(np.meshgrid(*axes, indexing='ij'), -1).reshape(-1, len(lo))
    if len(lo) == 2:
        if height is None:
            raise ValueError("a 2-D grid needs the antennas' height")
        points = np.concatenate([points, np.full((len(points), 1), float(height))], axis=1)
    return points

In [ ]:
g = grid_points((0, 0), (1, 0.5), 0.25, height=0.4)
assert g.shape == (5 * 3, 3) and (g[:, 2] == 0.4).all() and g[-1].tolist() == [1.0, 0.5, 0.4]
assert grid_points((0, 0, 0), (0.1, 0.1, 0.1), 0.05).shape == (27, 3)

In [ ]:
#| export
_C = 299_792_458.0                                     # speed of light (m/s)


class RayTracedPathsFading(Fading):
    "Paths ray-traced between anchor points, moved to the nodes' exact positions: continuous positions."

    def __init__(
        self,
        tx_points, # Transmitter anchors, (N_tx, 3)
        rx_points, # Receiver anchors, (N_rx, 3)
        a, # Complex gain of each path, (N_tx, N_rx, P); 0: no path
        tau, # Delay of each path (s), (N_tx, N_rx, P)
        u_tx, # Unit vector in which each path leaves the transmitter, (N_tx, N_rx, P, 3)
        u_rx, # Unit vector from the receiver towards where each path arrives from, (N_tx, N_rx, P, 3)
        frequency: float, # Carrier frequency (Hz)
        subcarriers=None, # Frequencies of the subcarriers relative to the carrier (Hz); None: the carrier alone
        reciprocal: bool = True, # Links go both ways: a path is also used from its receiver to its transmitter
        max_distance: float | None = None, # Farthest a node may be from its anchor; beyond it, an error (None: no limit)
        antenna_height: float | None = None, # z of the nodes given 2-D positions
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
    ):
        self.tx_points = np.asarray(tx_points, np.float64).reshape(-1, 3)
        self.rx_points = np.asarray(rx_points, np.float64).reshape(-1, 3)
        self.a = np.asarray(a, np.complex128)
        self.tau = np.asarray(tau, np.float64)
        self.u_tx = np.asarray(u_tx, np.float64)
        self.u_rx = np.asarray(u_rx, np.float64)
        shape = (len(self.tx_points), len(self.rx_points), self.a.shape[-1])
        for name, arr, s in (('a', self.a, shape), ('tau', self.tau, shape),
                             ('u_tx', self.u_tx, shape + (3,)), ('u_rx', self.u_rx, shape + (3,))):
            if arr.shape != s:
                raise ValueError(f"{name}: {s} for {shape[0]} transmitter and {shape[1]} receiver anchors, not {arr.shape}")
        self.frequency = float(frequency)
        self.subcarriers = np.zeros(1) if subcarriers is None else np.asarray(subcarriers, np.float64).reshape(-1)
        self.reciprocal, self.max_distance, self.antenna_height = reciprocal, max_distance, antenna_height
        self.node_positions: dict = {}
        self.set_positions(node_positions or {})

    @classmethod
    def from_npz(
        cls,
        path: str, # .npz written by `save`
        **kwargs, # The other arguments of `RayTracedPathsFading`
    ) -> RayTracedPathsFading:
        "Load ray-traced paths saved with `save`."
        with np.load(path) as f:
            arrays = {k: f[k] for k in ('tx_points', 'rx_points', 'a', 'tau', 'u_tx', 'u_rx')}
            frequency = float(f['frequency'])
        return cls(**arrays, frequency=frequency, **kwargs)

    def save(
        self,
        path: str, # Where to write the .npz
    ) -> None:
        "Save the paths (e.g. traced on a GPU node, used anywhere)."
        np.savez_compressed(path, tx_points=self.tx_points, rx_points=self.rx_points, a=self.a, tau=self.tau,
                            u_tx=self.u_tx, u_rx=self.u_rx, frequency=self.frequency)

    def set_positions(
        self,
        positions: Mapping[NodeId, Sequence], # Position of (some of) the nodes: (x, y, z), or (x, y) with `antenna_height`
    ) -> None:
        "Where the nodes are now."
        for node, p in positions.items():
            p = np.asarray(p, np.float64).reshape(-1)
            if p.shape[0] == 2:
                if self.antenna_height is None:
                    raise ValueError(f"node {node!r} has a 2-D position: give the antenna_height")
                p = np.append(p, self.antenna_height)
            if p.shape[0] != 3:
                raise ValueError(f"node {node!r}: a position (x, y) or (x, y, z), not {tuple(p)}")
            self.node_positions[node] = p

    def _position(self, node: NodeId) -> np.ndarray:
        if node not in self.node_positions:
            raise KeyError(f"no position for node {node!r}: call set_positions first")
        return self.node_positions[node]

    @staticmethod
    def _nearest(points: np.ndarray, p: np.ndarray) -> tuple:
        d2 = ((points - p) ** 2).sum(-1)
        i = int(np.argmin(d2))
        return i, float(np.sqrt(d2[i]))

    def _paths(self, sender: NodeId, receiver: NodeId) -> tuple:
        "The anchors' paths of a link and how far each end is from its anchor; None if both ends are on the same anchor."
        ps, pr = self._position(sender), self._position(receiver)
        i, di = self._nearest(self.tx_points, ps)
        j, dj = self._nearest(self.rx_points, pr)
        forward = (max(di, dj), i, j, False)
        best = forward
        if self.reciprocal:                            # the receiver at a transmitter anchor, the path reversed
            i2, di2 = self._nearest(self.tx_points, pr)
            j2, dj2 = self._nearest(self.rx_points, ps)
            if max(di2, dj2) < forward[0]:
                best = (max(di2, dj2), i2, j2, True)
        far, i, j, reverse = best
        if self.max_distance is not None and far > self.max_distance:
            raise KeyError(f"link {sender!r} -> {receiver!r} is {far:.3f} m from the nearest anchors "
                           f"(max_distance {self.max_distance} m): no paths were ray-traced there")
        if np.array_equal(self.tx_points[i], self.rx_points[j]):
            return None                                # both on the same anchor: no paths were traced there
        a, tau = self.a[i, j], self.tau[i, j]
        if not reverse:                                # leaves the sender like the anchor path, arrives like it
            return a, tau, self.u_tx[i, j], ps - self.tx_points[i], self.u_rx[i, j], pr - self.rx_points[j]
        return a, tau, self.u_rx[i, j], ps - self.rx_points[j], self.u_tx[i, j], pr - self.tx_points[i]

    def cfr(
        self,
        sender: NodeId, # The transmitting node
        receiver: NodeId, # The receiving node
    ) -> np.ndarray:
        "Channel frequency response at the subcarriers (one value without subcarriers), at the nodes' exact positions."
        paths = self._paths(sender, receiver)
        if paths is None:
            return np.full(len(self.subcarriers), complex(math.inf))
        a, tau, u_dep, d_dep, u_arr, d_arr = paths
        live = a != 0
        a, tau, u_dep, u_arr = a[live], tau[live], u_dep[live], u_arr[live]
        length = tau * _C
        # the transmitter's move, mirrored like the path (u_dep onto -u_arr): its image's move
        n = u_dep + u_arr
        nn = (n * n).sum(-1, keepdims=True)
        image_move = d_dep - np.divide(2 * n * (n @ d_dep)[:, None], nn, out=np.zeros_like(n), where=nn > 1e-12)
        moved = np.linalg.norm(length[:, None] * u_arr + image_move - d_arr, axis=-1)
        dtau = (moved - length) / _C
        a = a * (length / moved) * np.exp(-2j * np.pi * self.frequency * dtau)
        return np.exp(-2j * np.pi * np.outer(self.subcarriers, tau + dtau)) @ a

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain mean |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        if self._colocated(sender, receiver):
            return math.inf
        return float(np.mean(np.abs(self.cfr(sender, receiver)) ** 2))

    def _colocated(self, sender: NodeId, receiver: NodeId) -> bool:
        "At the same place, or nearer each other than the anchors (on the same anchor): a perfect link."
        return np.array_equal(self._position(sender), self._position(receiver)) or self._paths(sender, receiver) is None

    def response(self, transmission, link):
        "`cfr` of a transmission: H at each subcarrier (math.inf if co-located)."
        if self._colocated(transmission.sender, transmission.receiver):
            return np.full(len(self.subcarriers), complex(math.inf))
        return self.cfr(transmission.sender, transmission.receiver)

    def coefficient(self, transmission, link):
        "H at the carrier; with subcarriers, √(mean |H|²) (the same power). math.inf if co-located."
        if self._colocated(transmission.sender, transmission.receiver):
            return math.inf
        H = self.cfr(transmission.sender, transmission.receiver)
        return complex(H[0]) if H.shape[0] == 1 else complex(np.sqrt(np.mean(np.abs(H) ** 2)))

In [ ]:
show_doc(RayTracedPathsFading.cfr)

In [ ]:
show_doc(RayTracedPathsFading.set_positions)

In [ ]:
show_doc(RayTracedPathsFading.save)

In [ ]:
show_doc(RayTracedPathsFading.from_npz)

#### Tests

A geometry with an exact answer: a line-of-sight path and a reflection off a wall at `y = 3` (image method),
at 5.18 GHz. The paths are "ray-traced" at anchors every 5 cm, then the channel is computed at positions between
them and compared with the exact one.

In [ ]:
F = 5.18e9
WALL, GAMMA = 3.0, -0.7                                          # wall y = 3, reflection coefficient

def two_ray(pt, pr):
    "Exact paths from pt to pr: line of sight and the wall's reflection (image method)."
    pt, pr = np.asarray(pt, float), np.asarray(pr, float)
    mirror = lambda p: p * [1, -1, 1] + [0, 2 * WALL, 0]
    a, tau, ut, ur = [], [], [], []
    for gain, src, dst in ((1.0, pt, pr), (GAMMA, mirror(pt), pr)):
        length = np.linalg.norm(dst - src)
        a.append(gain * _C / F / (4 * np.pi * length) * np.exp(-2j * np.pi * F * length / _C))
        tau.append(length / _C)
        ur.append((src - dst) / length)                                  # towards the (image) source
    ut = [(pr - pt) / np.linalg.norm(pr - pt), (mirror(pr) - pt) / np.linalg.norm(mirror(pr) - pt)]
    return np.array(a), np.array(tau), np.array(ut), np.array(ur)

NO_PATHS = (np.zeros(2, complex), np.zeros(2), np.zeros((2, 3)), np.zeros((2, 3)))   # a point to itself

def trace(tx_points, rx_points):
    "Our 'ray tracer': the exact paths between every pair of anchors."
    out = {k: [] for k in ('a', 'tau', 'u_tx', 'u_rx')}
    for pt in tx_points:
        rows = [two_ray(pt, pr) if not np.array_equal(pt, pr) else NO_PATHS for pr in rx_points]
        for k, n in zip(out, range(4)):
            out[k].append([r[n] for r in rows])
    return {k: np.array(v) for k, v in out.items()}

exact_H = lambda pt, pr, f=np.zeros(1): np.exp(-2j * np.pi * np.outer(f, two_ray(pt, pr)[1])) @ two_ray(pt, pr)[0]

ap = np.array([[0.0, 0.0, 2.4]])                                # an access point
anchors = grid_points((2, 0), (3, 1), 0.05, height=0.4)         # a robot's area, every 5 cm
fading = RayTracedPathsFading(ap, anchors, **trace(ap, anchors), frequency=F, max_distance=0.1)
rng = np.random.default_rng(0)
robots = rng.uniform((2, 0), (3, 1), size=(200, 2))
err_plane, err_nearest = [], []
for xy in robots:
    pr = np.append(xy, 0.4)
    fading.set_positions({'ap': ap[0], 'robot': pr})
    truth = exact_H(ap[0], pr)[0]
    err_plane.append(abs(fading.cfr('ap', 'robot')[0] - truth) ** 2 / abs(truth) ** 2)
    nearest = anchors[np.argmin(((anchors - pr) ** 2).sum(-1))]
    err_nearest.append(abs(exact_H(ap[0], nearest)[0] - truth) ** 2 / abs(truth) ** 2)
print(f'error of H between 5 cm anchors: moved paths {10 * np.log10(np.mean(err_plane)):.0f} dB, '
      f'nearest anchor {10 * np.log10(np.mean(err_nearest)):.0f} dB')
assert np.mean(err_plane) < 1e-3 and np.mean(err_nearest) > 0.1
# at an anchor: exactly the traced channel; the reverse direction (robot -> access point) is the same channel
fading.set_positions({'robot': anchors[7]})
assert np.isclose(fading.cfr('ap', 'robot')[0], exact_H(ap[0], anchors[7])[0], rtol=1e-12)
assert np.isclose(fading.cfr('robot', 'ap')[0], fading.cfr('ap', 'robot')[0], rtol=1e-12)
t = Transmission(message=None, sender='ap', receiver='robot', start_time=0.0)
assert fading.response(t, None).shape == (1,) and NoFading(2j).response(t, None).tolist() == [2j]

In [ ]:
# both ends moving: agent-to-agent links, anchors every 10 cm on both sides
pts = grid_points((0, 0), (1, 1), 0.1, height=0.4)
agents = RayTracedPathsFading(pts, pts, **trace(pts, pts), frequency=F, antenna_height=0.4)
errs = []
for _ in range(200):
    p1, p2 = rng.uniform(0, 1, (2, 2))
    agents.set_positions({'a': p1, 'b': p2})                    # 2-D positions: at the antenna height
    if np.linalg.norm(p1 - p2) < 0.2:
        continue
    truth = exact_H(np.append(p1, 0.4), np.append(p2, 0.4))[0]
    errs.append(abs(agents.cfr('a', 'b')[0] - truth) ** 2 / abs(truth) ** 2)
print(f'agent-to-agent, 10 cm anchors on both ends: error {10 * np.log10(np.mean(errs)):.0f} dB')
assert np.mean(errs) < 1e-2
# wideband: 52 subcarriers of a 20 MHz channel
sc = (np.arange(52) - 25.5) * 312.5e3
wide = RayTracedPathsFading(ap, anchors, **trace(ap, anchors), frequency=F, subcarriers=sc)
pr = np.array([2.512, 0.333, 0.4])
wide.set_positions({'ap': ap[0], 'robot': pr})
H = wide.cfr('ap', 'robot')
assert H.shape == (52,) and np.allclose(H, exact_H(ap[0], pr, sc), rtol=0.05, atol=0)
assert abs(wide.path_gain('ap', 'robot') - np.mean(np.abs(exact_H(ap[0], pr, sc)) ** 2)) < 0.05 * wide.path_gain('ap', 'robot')

In [ ]:
# errors: no position, too far from every anchor, 2-D without a height; co-located; save / load
t = Transmission(message=None, sender='ap', receiver='robot', start_time=0.0)
fading.set_positions({'robot': (9.0, 9.0, 0.4)})
try:
    fading.coefficient(t, None); raise AssertionError('no error')
except KeyError as e:
    assert 'max_distance' in str(e)
try:
    fading.set_positions({'x': (1.0, 2.0)}); raise AssertionError('no error')
except ValueError as e:
    assert 'antenna_height' in str(e)
fading.set_positions({'robot': ap[0]})
assert fading.coefficient(t, None) == math.inf
agents.set_positions({'a': (0.31, 0.52), 'b': (0.29, 0.49)})       # 3.6 cm apart, on the same 10 cm anchor
assert agents.path_gain('a', 'b') == math.inf and np.isinf(agents.cfr('a', 'b')).all()
with tempfile.TemporaryDirectory() as tmp:
    fading.save(os.path.join(tmp, 'paths.npz'))
    loaded = RayTracedPathsFading.from_npz(os.path.join(tmp, 'paths.npz'), node_positions={'ap': ap[0], 'robot': (2.51, 0.3, 0.4)})
    fading.set_positions({'robot': (2.51, 0.3, 0.4)})
    assert np.array_equal(loaded.cfr('ap', 'robot'), fading.cfr('ap', 'robot'))

### In a channel

A ray-traced fading at the system level, then at the link level. 3 positions on a line: (0, 0) - (1, 0) - (5, 0);
the path gain falls with the distance.

In [ ]:
from comm_core.core import Message
from comm_core.net.topology import Link
from comm_core.radio.system.channel import SystemLevelChannel

In [ ]:
positions = [(0, 0), (1, 0), (5, 0)]
gain_db = np.array([[0, -60, -95], [-60, 0, -90], [-95, -90, 0]], float)           # H[rx, tx]
H = np.sqrt(10 ** (gain_db / 10)) * np.exp(1j * np.arange(9).reshape(3, 3))
np.fill_diagonal(H, 0)
table = RayTracedFading(H, positions, node_positions={'a': (0, 0), 'b': (1, 0), 'c': (5, 0)})
ch = SystemLevelChannel(fading=table, noise_power=1e-13, seed=0)                  # -100 dBm
assert abs(10 * np.log10(table.path_gain('a', 'b')) + 60) < 1e-9 and abs(10 * np.log10(table.path_gain('a', 'c')) + 95) < 1e-9

def send(ch, sender, receiver, bits=256, n=200):
    t = Transmission(message=Message(sender, receiver, payload=1, size_bits=bits), sender=sender, receiver=receiver, start_time=0.0)
    results = [ch.transmit(t, Link(sender, receiver, tx_power_w=1e-4)) for _ in range(n)]    # 0.1 mW
    return sum(r.success for r in results) / n, results[0]

near, r = send(ch, 'a', 'b')                               # 0.1 mW x -60 dB / -100 dBm: 30 dB, always received
far, _ = send(ch, 'a', 'c')                                # -5 dB: lost
assert near == 1.0 and far == 0.0 and abs(r.snr_db - 30) < 1e-9 and abs(r.metadata['channel_gain_db'] + 60) < 1e-9
# the nodes move (told to the channel, which tells its fading): the channel follows
ch.set_positions({'c': (1, 0)})
assert send(ch, 'a', 'c')[0] == 1.0
ch.set_positions({'b': (0, 0)})
assert table.path_gain('a', 'b') == math.inf and send(ch, 'a', 'b')[0] == 1.0     # same cell
# in between: losses, reproducible with the seed
ch = SystemLevelChannel(fading=RayTracedFading(H, positions, node_positions={'a': (0, 0), 'b': (1, 0)}), noise_power=2e-11)   # 7 dB
ch.reset(seed=3); first = [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)]
ch.reset(seed=3); assert [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)] == first
assert 0 < first[0] < 1
# the threshold model; a node where nothing was ray-traced
ch = SystemLevelChannel(fading=RayTracedFading(H, positions, node_positions={'a': (0, 0), 'c': (5, 0)}), noise_power=1e-13,
                        per_model='threshold', snr_threshold_db=-10)
assert send(ch, 'a', 'c')[0] == 1.0                                            # -5 dB, above the threshold
ch.set_positions({'c': (9, 9)})
try:
    send(ch, 'a', 'c'); raise AssertionError('no error')
except KeyError as e:
    assert 'no channel was ray-traced' in str(e)

In a `Network`, the application moves the nodes before each step:

In [ ]:
from comm_core import Network, Node, FullMeshTopology, IdealProtocol, AnalyticalBackend

In [ ]:
ch = SystemLevelChannel(fading=RayTracedFading(H, positions), noise_power=1e-8, seed=0)   # 1 W links: 20 dB at (1, 0), -15 dB at (5, 0)
net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(ch))
received = []
for pos_b in [(1, 0), (5, 0), (1, 0)]:                  # b walks away and comes back
    ch.set_positions({'a': (0, 0), 'b': pos_b})
    net.send(Message('a', 'b', payload='hi', size_bits=1024))
    net.step()
    received.append(len(net.receive('b')))
assert received == [1, 0, 1]

The same table at the link level. Every bit is simulated, and channel inversion uses just the power each link
needs:

In [ ]:
from comm_core.radio.link.channel import LinkLevelChannel
from comm_core.radio.power_control import ChannelInversion

In [ ]:
table = RayTracedFading(H, positions)
ch = LinkLevelChannel(fading=table, power_control=ChannelInversion(target_snr_db=10, max_power=1.0), noise_power=1e-13, seed=0)
nodes = ['a', 'b', 'c']
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), IdealProtocol(), AnalyticalBackend(ch))
ch.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (5, 0)})      # passed on to the table
for s in nodes:
    for r in nodes:
        if s != r:
            net.send(Message(s, r, payload=np.arange(8)))
results = {(x.transmission.sender, x.transmission.receiver): x for x in net.step()}
# 10 dB over -100 dBm: a-b (-60 dB) needs 1e-6 W, b-c (-90 dB) 1e-3 W, a-c (-95 dB) 3.2e-3 W, all within 1 W
assert all(x.success for x in results.values())
assert abs(results['a', 'b'].metadata['tx_power'] - 1e-6) < 1e-12 and abs(results['a', 'c'].snr_db - 10) < 1e-6
ch.power_control.max_power = 2e-3                              # 2 mW: a-c is not used
results = {(s, r): ch.transmit(Transmission(Message(s, r, np.arange(8)), s, r, 0.0), Link(s, r)).success
           for s in nodes for r in nodes if s != r}
assert results == {('a', 'b'): True, ('b', 'a'): True, ('b', 'c'): True, ('c', 'b'): True, ('a', 'c'): False, ('c', 'a'): False}

With a channel, a robot moving continuously through the area:

In [ ]:
channel = SystemLevelChannel(fading=fading, noise_power=10 ** ((-94 - 30) / 10), seed=0)   # -94 dBm noise, 1 W
snr = []
for x in np.linspace(2.0, 3.0, 11):
    channel.set_positions({'ap': ap[0], 'robot': (x, 0.37, 0.4)})
    r = channel.transmit(Transmission(Message('ap', 'robot', None, size_bits=1000), 'ap', 'robot', 0.0), Link('ap', 'robot'))
    snr.append(round(r.snr_db, 1))
print('SNR along the way (dB):', snr)
assert all(s > 0 for s in snr)

## Tracing the paths

`RayTracedFading` takes a channel at fixed points, which suits discrete environments. For
continuous ones, `trace_paths` ray-traces the *paths* between anchor points in a Sionna RT scene, e.g. every 5 cm
(`grid_points`). `RayTracedPathsFading` (see `fading`) then moves them to wherever the nodes are.

Trace once, where Sionna RT runs (e.g. a GPU node), and save:

```python
from sionna.rt import load_scene
scene = load_scene('my_room.xml')
scene.frequency = 5.18e9
access_point = [[0.0, 0.0, 2.4]]
robot_area = grid_points((-6.5, -3.0), (-3.0, -0.5), 0.05, height=0.4)      # every 5 cm, antennas at 0.4 m
paths = trace_paths(scene, access_point, robot_area, max_depth=6)
RayTracedPathsFading(**paths).save('room_paths.npz')
```

Then use the saved paths anywhere, without Sionna:

```python
fading = RayTracedPathsFading.from_npz('room_paths.npz', max_distance=0.1)
channel = SystemLevelChannel(fading=fading, noise_power=..., ...)
channel.set_positions({'ap': (0, 0, 2.4), 'robot': (-4.213, -1.871, 0.4)})   # anywhere in the area
```

**Agent-to-agent links** use the same points on both sides: `trace_paths(scene, points, points)`.

**What it costs.** The table holds every transmitter anchor × receiver anchor × path:

* **Access point to robot:** one transmitter, so anchors every 5 cm are cheap. A 10 × 10 m area is 40k anchors,
  about 100 MB with 30 paths.
* **Agent to agent:** the number of pairs grows with the square of the number of anchors, so anchors every 5 cm
  are only possible over small areas. Moving the paths, rather than taking the nearest anchor's channel, is what
  lets the anchors be further apart: see the validation below.

Coincident transmitter and receiver anchors get no paths, so nodes that share an anchor have a perfect link (see
`RayTracedPathsFading`).

In [ ]:
#| export
def _unit(theta, phi):
    "Unit vectors of zenith / azimuth angles."
    return np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], -1)


def trace_paths(
    scene, # A Sionna RT scene with its frequency set; its tx_array / rx_array, or one isotropic antenna each
    tx_points, # Transmitter anchors, (N_tx, 3)
    rx_points, # Receiver anchors, (N_rx, 3)
    batch: int = 8, # Transmitters, and receivers, per PathSolver call (more overflow its path buffer and drop paths)
    progress: bool = False, # Print the progress
    **solver_kwargs, # PathSolver options; default max_depth=3, line of sight, specular reflections, refraction
) -> dict:
    "Ray-trace the paths between every transmitter and receiver anchor: the arguments of `RayTracedPathsFading`."
    import mitsuba as mi
    from sionna.rt import PathSolver, PlanarArray, Receiver, Transmitter
    tx_points = np.asarray(tx_points, np.float64).reshape(-1, 3)
    rx_points = np.asarray(rx_points, np.float64).reshape(-1, 3)
    if scene.tx_array is None:
        scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern='iso', polarization='V')
    if scene.rx_array is None:
        scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern='iso', polarization='V')
    kwargs = dict(max_depth=3, los=True, specular_reflection=True, diffuse_reflection=False, refraction=True,
                  synthetic_array=True) | solver_kwargs
    solver = PathSolver()
    own = [*scene.transmitters.values(), *scene.receivers.values()]     # the scene's radios, put back afterwards
    for radio in own:
        scene.remove(radio.name)
    point = lambda p: mi.Point3f(*map(float, p))
    blocks, num_paths = [], 0
    calls = -(-len(tx_points) // batch) * -(-len(rx_points) // batch)
    try:
        for i0 in range(0, len(tx_points), batch):
            for j0 in range(0, len(rx_points), batch):
                tx, rx = tx_points[i0:i0 + batch], rx_points[j0:j0 + batch]
                for k, p in enumerate(tx):
                    scene.add(Transmitter(f'cc-tx-{k}', position=point(p)))
                for k, p in enumerate(rx):
                    scene.add(Receiver(f'cc-rx-{k}', position=point(p)))
                paths = solver(scene, **kwargs)
                a, tau = paths.cir(normalize_delays=False, out_type='numpy')
                a = a[:, 0, :, 0, :, 0]                                   # (rx, tx, paths): one antenna each
                valid = paths.valid.numpy().astype(bool) & (np.abs(a) > 0) & np.isfinite(a) & np.isfinite(tau)
                valid &= ~(np.abs(rx[:, None, None] - tx[None, :, None]).sum(-1) < 1e-9)   # coincident anchors
                u_tx = _unit(paths.theta_t.numpy(), paths.phi_t.numpy())
                u_rx = _unit(paths.theta_r.numpy(), paths.phi_r.numpy())
                sw = lambda x: np.swapaxes(x, 0, 1)                       # (tx, rx, ...)
                blocks.append((i0, j0, sw(np.where(valid, a, 0)), sw(np.where(valid, tau, 0)),
                               sw(np.where(valid[..., None], u_tx, 0)), sw(np.where(valid[..., None], u_rx, 0))))
                num_paths = max(num_paths, a.shape[-1])
                for k in range(len(tx)):
                    scene.remove(f'cc-tx-{k}')
                for k in range(len(rx)):
                    scene.remove(f'cc-rx-{k}')
                if progress:
                    print(f'{len(blocks)} / {calls}', end='\r')
    finally:
        for name in [n for n in (*scene.transmitters, *scene.receivers) if n.startswith('cc-')]:
            scene.remove(name)
        for radio in own:
            scene.add(radio)
    shape = (len(tx_points), len(rx_points), num_paths)
    out = dict(a=np.zeros(shape, np.complex128), tau=np.zeros(shape), u_tx=np.zeros(shape + (3,)), u_rx=np.zeros(shape + (3,)))
    for i0, j0, a, tau, u_tx, u_rx in blocks:
        at = (slice(i0, i0 + a.shape[0]), slice(j0, j0 + a.shape[1]), slice(0, a.shape[-1]))
        out['a'][at], out['tau'][at], out['u_tx'][at], out['u_rx'][at] = a, tau, u_tx, u_rx
    frequency = float(np.asarray(scene.frequency).reshape(-1)[0])
    return dict(tx_points=tx_points, rx_points=rx_points, **out, frequency=frequency)

In [ ]:
show_doc(trace_paths)

### Validation against Sionna RT

The test: Sionna's street-canyon scene at 5.18 GHz, with reflections off the buildings and the ground (up to 3
bounces). The channel at 40 random positions between the anchors, moved from the anchors, is compared with the
channel ray-traced directly at those positions, over 52 subcarriers (20 MHz). For reference, two other ways are
shown: plane waves (`Δτ = −(u_tx·Δ_tx + u_rx·Δ_rx)/c`, as in a first-order extrapolation) and the nearest
anchor's channel. Lower is better; −20 dB means errors of 1% of the channel's power.

This needs Sionna RT, so it does not run in the tests; the output was saved when it ran (Sionna 2.2 with Sionna RT 2.2, on a CPU).

In [ ]:
#| eval: false
import sionna.rt as rt
from sionna.rt import load_scene

scene = load_scene(rt.scene.simple_street_canyon)
scene.frequency = 5.18e9
subcarriers = (np.arange(52) - 25.5) * 312.5e3                      # 20 MHz, 52 subcarriers
rng = np.random.default_rng(0)

def response(a, tau):
    return np.exp(-2j * np.pi * np.outer(subcarriers, tau)) @ a

def plane_waves(fading, s, r):
    "For comparison, the first-order move: Δτ = −(u_tx·Δ_tx + u_rx·Δ_rx) / c."
    a, tau, u_dep, d_dep, u_arr, d_arr = fading._paths(s, r)
    live = a != 0
    dtau = -(u_dep[live] @ d_dep + u_arr[live] @ d_arr) / 299_792_458.0
    return response(a[live] * np.exp(-2j * np.pi * fading.frequency * dtau), tau[live] + dtau)

def compare(label, tx_anchors, rx_anchors, pairs):
    fading = RayTracedPathsFading(**trace_paths(scene, tx_anchors, rx_anchors), subcarriers=subcarriers)
    errors = {'moved paths': [], 'plane waves': [], 'nearest anchor': []}
    power, same_paths = [], []
    for pt, pr in pairs:
        direct = trace_paths(scene, pt[None], pr[None])                # the truth: traced right there
        truth = response(direct['a'][0, 0], direct['tau'][0, 0])
        fading.set_positions({'s': pt, 'r': pr})
        i = np.argmin(((fading.tx_points - pt) ** 2).sum(-1))
        j = np.argmin(((fading.rx_points - pr) ** 2).sum(-1))
        for name, estimate in (('moved paths', fading.cfr('s', 'r')), ('plane waves', plane_waves(fading, 's', 'r')),
                               ('nearest anchor', response(fading.a[i, j], fading.tau[i, j]))):
            errors[name].append(np.sum(np.abs(estimate - truth) ** 2))
        power.append(np.sum(np.abs(truth) ** 2))
        same_paths.append((fading.a[i, j] != 0).sum() == (direct['a'][0, 0] != 0).sum())
    nmse = lambda e, keep: 10 * np.log10(np.sum(np.array(e)[keep]) / np.sum(np.array(power)[keep]))
    everything, same = np.ones(len(pairs), bool), np.array(same_paths)
    print(f'{label}: ' + ', '.join(f'{name} {nmse(e, everything):.1f} dB' for name, e in errors.items()))
    if not same.all():
        print(f'    {(~same).sum()} of {len(pairs)} positions have other paths than their anchors; '
              f'without them, moved paths {nmse(errors["moved paths"], same):.1f} dB')

access_point = np.array([[-30.0, 0.0, 6.0]])
lo, hi = (10.0, -1.0), (11.0, 0.0)
robots = [(access_point[0], np.append(rng.uniform(lo, hi), 1.5)) for _ in range(40)]
for spacing in (0.05, 0.2, 0.5):
    compare(f'access point -> robot, anchors every {spacing * 100:.0f} cm', access_point,
            grid_points(lo, hi, spacing, height=1.5), robots)
area_a, area_b = ((-5.0, -1.0), (-4.0, 0.0)), ((5.0, -1.0), (6.0, 0.0))
agents = [(np.append(rng.uniform(*area_a), 1.5), np.append(rng.uniform(*area_b), 1.5)) for _ in range(30)]
compare('agent -> agent, anchors every 25 cm', grid_points(*area_a, 0.25, height=1.5),
        grid_points(*area_b, 0.25, height=1.5), agents)

access point -> robot, anchors every 5 cm: moved paths -62.3 dB, plane waves -60.5 dB, nearest anchor 2.0 dB
access point -> robot, anchors every 20 cm: moved paths -56.8 dB, plane waves -44.2 dB, nearest anchor 2.4 dB
access point -> robot, anchors every 50 cm: moved paths -47.5 dB, plane waves -29.4 dB, nearest anchor 2.9 dB
agent -> agent, anchors every 25 cm: moved paths -29.1 dB, plane waves -20.9 dB, nearest anchor 2.4 dB
    4 of 30 positions have other paths than their anchors; without them, moved paths -43.5 dB


* **A fixed access point:** moving the paths keeps the channel within −47 to −63 dB of the one traced directly,
  even with anchors 50 cm apart.
* **Plane waves** need denser anchors for the same accuracy.
* **The nearest anchor's channel** is unusable at any spacing: at a few centimetres from the anchor, its phase is
  already unrelated to the true one.
* **Both ends moving:** at some positions a reflection appears or disappears, e.g. at a building's edge, so the
  paths there differ from the anchors' paths. Those positions set the average error. Elsewhere it is −43.5 dB, and
  denser anchors only help near such edges.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()